# ⚾ Pitch Mix Dashboard – Data Pipeline

Ten notebook pobiera dane ze Statcast przez `pybaseball`, czyści je
i zapisuje do `data/pitch_mix_clean.parquet` — gotowego do wczytania przez `app.py`.

**Uruchom raz przed startem dashboardu.**

---
## Architektura

```
Statcast API (pybaseball)
       │
       ▼
[Krok 1] Pobieranie tygodniami  →  data/statcast_raw.parquet
       │
       ▼
[Krok 2] Czyszczenie + mapowanie ID→nazwiska
       │
       ▼
[Krok 3] EDA i wykresy diagnostyczne
       │
       ▼
[Krok 4] Eksport  →  data/pitch_mix_clean.parquet
       │
       ▼
app.py  wczytuje plik w ~0.2s  ✅
```


In [ ]:
# ── Instalacja (odkomentuj przy pierwszym uruchomieniu) ──
# !pip install pybaseball pandas pyarrow matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from datetime import datetime, timedelta

try:
    from pybaseball import statcast, playerid_reverse_lookup, cache
    cache.enable()   # lokalne cache pybaseball
    PYBASEBALL_OK = True
    print('✅ pybaseball zaladowany')
except ImportError:
    PYBASEBALL_OK = False
    print('❌ pip install pybaseball')

OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Folder: {OUTPUT_DIR.resolve()}')


## Krok 1 — Pobieranie danych ze Statcast

Pobieramy tygodniami, żeby uniknąć timeoutów. Pełny sezon ~15 min.
> Ogranicz do `2024-06-01 → 2024-08-31` na szybki test.


In [ ]:
SEASON_START = '2024-04-01'
SEASON_END   = '2024-09-29'
RAW_FILE = OUTPUT_DIR / 'statcast_raw.parquet'

if RAW_FILE.exists():
    size_mb = RAW_FILE.stat().st_size / 1_048_576
    print(f'Plik juz istnieje ({size_mb:.1f} MB): {RAW_FILE}')
elif PYBASEBALL_OK:
    chunks, cur = [], datetime.strptime(SEASON_START, '%Y-%m-%d')
    end  = datetime.strptime(SEASON_END,   '%Y-%m-%d')
    total_weeks = ((end - cur).days // 7) + 1
    week_num = 0
    while cur <= end:
        week_end = min(cur + timedelta(days=6), end)
        week_num += 1
        print(f'[{week_num:>2}/{total_weeks}] {cur:%Y-%m-%d} -> {week_end:%Y-%m-%d}', end=' ')
        try:
            chunk = statcast(start_dt=cur.strftime('%Y-%m-%d'),
                             end_dt=week_end.strftime('%Y-%m-%d'))
            if chunk is not None and len(chunk) > 0:
                chunks.append(chunk)
                print(f'{len(chunk):,} pitchy')
            else:
                print('brak danych')
        except Exception as e:
            print(f'Blad: {e}')
        cur = week_end + timedelta(days=1)
    df_raw = pd.concat(chunks, ignore_index=True)
    df_raw.to_parquet(RAW_FILE, index=False)
    print(f'Zapisano {len(df_raw):,} pitchy -> {RAW_FILE}')
else:
    print('Pobierz CSV ze: https://baseballsavant.mlb.com/statcast_search')
    print('Zapisz jako data/statcast_raw.csv')


## Krok 2 — Wczytanie, czyszczenie, mapowanie nazw

In [ ]:
if RAW_FILE.exists():
    df_raw = pd.read_parquet(RAW_FILE)
else:
    csv_path = OUTPUT_DIR / 'statcast_raw.csv'
    df_raw = pd.read_csv(csv_path, low_memory=False)

print(f'Wierszy: {len(df_raw):,}  |  Kolumn: {df_raw.shape[1]}')
print(f'Zakres dat: {df_raw["game_date"].min()} -> {df_raw["game_date"].max()}')

# ── Wybor kolumn ──
KEEP = ['game_date','player_name','batter','pitcher',
        'pitch_type','stand','p_throws','events','description','inning']
df = df_raw[[c for c in KEEP if c in df_raw.columns]].copy()
df['game_date'] = pd.to_datetime(df['game_date'])

# ── Czyszczenie ──
before = len(df)
df = df.dropna(subset=['pitch_type','player_name','batter'])
df = df[df['pitch_type'].str.strip() != '']
df['batter']  = df['batter'].astype(int)
df['pitcher'] = df['pitcher'].astype(int)
print(f'Usunieto {before-len(df):,} wierszy  |  Pozostalo: {len(df):,}')

# ── Mapowanie ID batter -> nazwisko ──
if PYBASEBALL_OK:
    try:
        ids = df['batter'].unique().tolist()
        lkp = playerid_reverse_lookup(ids, key_type='mlbam')
        lkp['batter_name'] = lkp['name_first'] + ' ' + lkp['name_last']
        lkp = lkp[['key_mlbam','batter_name']].rename(columns={'key_mlbam':'batter'})
        df = df.merge(lkp, on='batter', how='left')
        print(f'Zmapowano {df["batter_name"].notna().sum():,} pitchy')
    except Exception as e:
        print(f'Blad lookup: {e}')
        df['batter_name'] = 'Batter #' + df['batter'].astype(str)
else:
    df['batter_name'] = 'Batter #' + df['batter'].astype(str)

df['batter_name'] = df['batter_name'].fillna('Unknown #' + df['batter'].astype(str))

# Statcast format nazwiska pitchera: 'Kowalski, Jan' -> 'Jan Kowalski'
def flip(name):
    if pd.isna(name) or ',' not in str(name): return name
    last, first = name.split(',', 1)
    return f'{first.strip()} {last.strip()}'

df['pitcher_name'] = df['player_name'].apply(flip)
df[['pitcher_name','batter_name','pitch_type','game_date','stand','p_throws']].head(5)


## Krok 3 — EDA (wykresy diagnostyczne)

In [ ]:
plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#e6edf3',
    'xtick.color':'#7d8590','ytick.color':'#7d8590',
    'text.color':'#e6edf3','grid.color':'#21262d',
    'grid.linestyle':'--','grid.alpha':0.6,
})
C = ['#ff6b35','#4cc9f0','#39d353','#ffd166','#bc8cff']

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Statcast EDA', fontsize=14, fontweight='bold', color='#ff6b35')

# 1. Rozkład pitch type
ax = axes[0,0]
pt = df['pitch_type'].value_counts().head(10)
bars = ax.barh(pt.index[::-1], pt.values[::-1], color=C[0], edgecolor='#0d1117')
ax.set_title('Rozkład pitch types', fontweight='bold')
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=8)

# 2. Wolumen tygodniowy
ax = axes[0,1]
df['week_start'] = df['game_date'].dt.to_period('W').apply(lambda p: p.start_time)
wv = df.groupby('week_start').size()
ax.fill_between(wv.index, wv.values, alpha=0.3, color=C[1])
ax.plot(wv.index, wv.values, color=C[1], lw=2)
ax.set_title('Pitchy tygodniowo', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))

# 3. Top 10 pitcherów
ax = axes[0,2]
top_p = df['pitcher_name'].value_counts().head(10)
ax.barh(top_p.index[::-1], top_p.values[::-1], color=C[2], edgecolor='#0d1117')
ax.set_title('Top 10 Pitcherów', fontweight='bold')

# 4. L vs R batter
ax = axes[1,0]
if 'stand' in df.columns:
    sc = df['stand'].value_counts()
    ax.pie(sc.values, labels=sc.index, colors=C[:2],
           autopct='%1.1f%%', textprops={'color':'#e6edf3'})
    ax.set_title('Batter: L vs R', fontweight='bold')

# 5. Pitch mix 1. vs 2. polowa
ax = axes[1,1]
df['half'] = (df['game_date'].dt.month >= 7).map({False:'1H',True:'2H'})
ph = df.groupby(['half','pitch_type']).size().unstack(fill_value=0)
ph_pct = ph.div(ph.sum(axis=1), axis=0)*100
top6 = df['pitch_type'].value_counts().head(6).index
ph_pct[top6].T.plot(kind='bar', ax=ax, color=C, edgecolor='#0d1117', width=0.6)
ax.set_title('Mix% — 1H vs 2H', fontweight='bold')
ax.set_ylabel('%'); ax.legend(fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

# 6. Matchups per pitcher
ax = axes[1,2]
mu = df.groupby('pitcher_name')['batter_name'].nunique().sort_values(ascending=False).head(10)
ax.barh(mu.index[::-1], mu.values[::-1], color=C[4], edgecolor='#0d1117')
ax.set_title('Top 10 Pitcherów — unikalne matchupy', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Wykres zapisany -> data/eda_overview.png')


## Krok 4 — Sanity check pitch mix

In [ ]:
example = df['pitcher_name'].value_counts().index[0]
p_df = df[df['pitcher_name'] == example].copy()
p_weekly = p_df.groupby(['week_start','pitch_type']).size().reset_index(name='n')
totals = p_weekly.groupby('week_start')['n'].transform('sum')
p_weekly['pct'] = p_weekly['n'] / totals * 100
pivot = p_weekly.pivot_table(index='pitch_type', columns='week_start',
                              values='pct', aggfunc='sum').fillna(0)

fig, ax = plt.subplots(figsize=(15, 5))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
for i, pt in enumerate(pivot.index):
    ax.plot(range(len(pivot.columns)), pivot.loc[pt], marker='o',
            lw=2, label=pt, color=C[i % len(C)])
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c.date()) for c in pivot.columns], rotation=35, ha='right', fontsize=8)
ax.set_title(f'Tygodniowy pitch mix — {example}', fontweight='bold', color='#e6edf3')
ax.set_ylabel('Udzial (%)', color='#e6edf3'); ax.grid(True)
ax.legend(loc='upper right', fontsize=9, framealpha=0.3)
plt.tight_layout(); plt.show()


## Krok 5 — Eksport finalnego pliku

In [ ]:
FINAL_COLS = ['game_date','pitcher_name','batter_name',
              'pitch_type','stand','p_throws']
df_export = df[[c for c in FINAL_COLS if c in df.columns]].copy()
df_export = df_export.dropna(subset=['pitcher_name','batter_name','pitch_type'])

FINAL_FILE = OUTPUT_DIR / 'pitch_mix_clean.parquet'
df_export.to_parquet(FINAL_FILE, index=False)

size_mb = FINAL_FILE.stat().st_size / 1_048_576
print('Eksport gotowy!')
print(f'  Plik:      {FINAL_FILE}')
print(f'  Wierszy:   {len(df_export):,}')
print(f'  Rozmiar:   {size_mb:.1f} MB')
print()
print('='*45)
print(f'  Pitchers : {df_export["pitcher_name"].nunique():>6,}')
print(f'  Batters  : {df_export["batter_name"].nunique():>6,}')
print(f'  Matchups : {df_export.groupby(["pitcher_name","batter_name"]).ngroups:>6,}')
print(f'  Tygodnie : {df_export["game_date"].dt.to_period("W").nunique():>6}')
print(f'  Pitche   : {len(df_export):>6,}')
print('='*45)


## Jak wczytać dane w `app.py`

Zamień `load_or_generate_data()` na:

```python
@st.cache_data(show_spinner=False)
def load_data() -> pd.DataFrame:
    parquet = Path('data/pitch_mix_clean.parquet')
    if parquet.exists():
        return pd.read_parquet(parquet)
    return _generate_synthetic_data('2024-04-01', '2024-09-29')

raw_df = load_data()
```

Dashboard wczyta plik w **~0.2 s** zamiast pobierać Statcast przy każdym starcie. 🚀
